Not: Bu dosya, model mimarisinin kurulması, veri yükleyicilerin (Dataloader) test edilmesi ve ilk prototipleme aşamaları için bir karalama defteri (sandbox) olarak kullanılmıştır. Bu nedenle hücre sıralamalarında farklı denemelerin izleri bulunabilir.

In [3]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
#bozuk dosyaları es geç
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
# --- 1. AYARLAR VE DOSYA YOLLARI ---
# Kendi bilgisayarındaki yollara göre buraları kontrol etmeyi unutma!
CSV_YOLU = r"C:\Users\emirh\Desktop\machlearn\faces\diyet_projesi\buyuk_veri\visual_bmi_annotations.csv"
RESIM_ANA_KLASOR = r"C:\Users\emirh\Desktop\machlearn\faces\diyet_projesi\buyuk_veri\Visual BMI"
MODEL_KAYIT_ADI = "bmi_model_buyuk_veri_v1.pth"

# Cihaz Seçimi (RTX 3080 Göreve!)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Kullanılan Cihaz: {device}")

# --- 2. VERİ YÜKLEYİCİ (DATASET) SINIFI ---
class BuyukBmiDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # CSV'deki /kaggle/input/... yolunu yerel yola çeviriyoruz
        goreceli_yol = self.df.iloc[idx]['image_path'].replace('/kaggle/input/visual-bmi/', '')
        img_path = os.path.join(self.root_dir, goreceli_yol).replace('/', os.sep)
        
        image = Image.open(img_path).convert("RGB")
        bmi = float(self.df.iloc[idx]['BMI'])
        
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(bmi, dtype=torch.float32)

# --- 3. MODEL MİMARİSİ (CNN) ---
class BmiNet(nn.Module):
    def __init__(self):
        super(BmiNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1) # 16'dan 32'ye çıkardık, büyük veri daha fazla özellik ister
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        
        self.fc1 = nn.Linear(128 * 28 * 28, 512)
        self.fc2 = nn.Linear(512, 128)
        self.fc3 = nn.Linear(128, 1)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(-1, 128 * 28 * 28)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

# --- 4. HAZIRLIK VE EĞİTİM BAŞLANGICI ---

# Veri Dönüşümleri (Normalizasyon eklendi)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

dataset = BuyukBmiDataset(csv_file=CSV_YOLU, root_dir=RESIM_ANA_KLASOR, transform=transform)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

model = BmiNet().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005) # Büyük veride daha kararlı öğrenme için lr'yi biraz düşürdük

# --- 5. EĞİTİM DÖNGÜSÜ ---
epochs = 20
print(f"Toplam {len(dataset)} fotoğraf ile {epochs} turluk maraton başlıyor...")

for epoch in range(epochs):
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device).view(-1, 1)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    print(f"Tur [{epoch+1}/{epochs}] - Ortalama Hata: {running_loss/len(train_loader):.4f}")

# --- 6. KAYDETME ---
torch.save(model.state_dict(), MODEL_KAYIT_ADI)
print(f"✅ Eğitim bitti! Final modeli '{MODEL_KAYIT_ADI}' olarak kaydedildi.")

Kullanılan Cihaz: cuda
Toplam 5897 fotoğraf ile 20 turluk maraton başlıyor...
Tur [1/20] - Ortalama Hata: 90.6434
Tur [2/20] - Ortalama Hata: 68.5529
Tur [3/20] - Ortalama Hata: 65.0247
Tur [4/20] - Ortalama Hata: 61.1548
Tur [5/20] - Ortalama Hata: 56.3527
Tur [6/20] - Ortalama Hata: 50.0193
Tur [7/20] - Ortalama Hata: 43.4153
Tur [8/20] - Ortalama Hata: 32.5414
Tur [9/20] - Ortalama Hata: 24.2890
Tur [10/20] - Ortalama Hata: 15.1574
Tur [11/20] - Ortalama Hata: 9.3740
Tur [12/20] - Ortalama Hata: 5.8399
Tur [13/20] - Ortalama Hata: 4.1394
Tur [14/20] - Ortalama Hata: 2.6779
Tur [15/20] - Ortalama Hata: 2.4335
Tur [16/20] - Ortalama Hata: 2.0671
Tur [17/20] - Ortalama Hata: 1.9360
Tur [18/20] - Ortalama Hata: 1.7647
Tur [19/20] - Ortalama Hata: 1.6321
Tur [20/20] - Ortalama Hata: 1.5740
✅ Eğitim bitti! Final modeli 'bmi_model_buyuk_veri_v1.pth' olarak kaydedildi.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import gradio as gr
from torchvision import transforms
from PIL import Image

# 1. CİHAZ TANIMLAMA (Hata buradaydı, tekrar ekliyoruz)
cihaz = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Arayüz şu cihazda çalışıyor: {cihaz}")

# 2. MODEL MİMARİSİ (Büyük veri eğitimindekiyle birebir aynı olmalı)
class BmiNet(nn.Module):
    def __init__(self):
        super(BmiNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.fc1 = nn.Linear(128 * 28 * 28, 512)
        self.fc2 = nn.Linear(512, 128)
        self.fc3 = nn.Linear(128, 1)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(-1, 128 * 28 * 28)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

# 3. MODELİ YÜKLEME
model_dosyasi = r"C:\Users\emirh\Desktop\machlearn\faces\diyet_projesi\bmi_model_buyuk_veri_v1.pth"
model_gui = BmiNet().to(cihaz) # Artık 'cihaz' tanımlı olduğu için hata vermeyecek
model_gui.load_state_dict(torch.load(model_dosyasi))
model_gui.eval()

# 4. TAHMİN FONKSİYONU VE GRADIO ARAYÜZÜ (Öncekiyle aynı)
donusum = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def predict_bmi(inp_image):
    img_tensor = donusum(inp_image).unsqueeze(0).to(cihaz)
    with torch.no_grad():
        tahmin = model_gui(img_tensor)
        bmi_sonuc = tahmin.item()
    
    return f"### 🤖 Yapay Zeka Tahmini\n**BMI Değeri:** {bmi_sonuc:.2f}"

# Arayüzü başlat
gr.Interface(fn=predict_bmi, inputs=gr.Image(type="pil"), outputs=gr.Markdown()).launch()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# --- 1. MODELİNDEN TAHMİNLERİ ALMA ---
# Bu kısım senin Buyukegitim.ipynb dosyasındaki test veri yükleyicine göre güncellenmelidir.
# Örnek: 'test_loader' adında bir test veri yükleyicin olduğunu varsayalım.
actual_bmis = []
predicted_bmis = []

model.eval() # Modeli değerlendirme moduna al
with torch.no_grad():
    for images, labels in test_loader: # 'test_loader' senin test veri yükleyicin
        images, labels = images.to(cihaz), labels.to(cihaz)
        outputs = model(images)
        actual_bmis.extend(labels.cpu().numpy())
        predicted_bmis.extend(outputs.cpu().numpy().flatten())

# Verileri numpy dizisine çevir
actual_bmis = np.array(actual_bmis)
predicted_bmis = np.array(predicted_bmis)

# --- 2. PERFORMANS METRİKLERİNİ HESAPLAMA ---
mse = mean_squared_error(actual_bmis, predicted_bmis)
rmse = np.sqrt(mse)
mae = mean_absolute_error(actual_bmis, predicted_bmis)
r2 = r2_score(actual_bmis, predicted_bmis)

# --- 3. SONUÇLARI YAZDIRMA ---
print("📊 Vücut Kitle İndeksi (BMI) Regresyon Performansı")
print("--------------------------------------------------")
print(f"Mean Squared Error (MSE)   : {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"Mean Absolute Error (MAE)  : {mae:.4f}")
print(f"R-squared (R2) Skor        : {r2:.4f}")
print("--------------------------------------------------")

# --- 4. TAHMİN vs GERÇEK GRAFİĞİNİ ÇİZME ---
plt.figure(figsize=(10, 8))
sns.scatterplot(x=actual_bmis, y=predicted_bmis, alpha=0.6)
plt.plot([actual_bmis.min(), actual_bmis.max()], [actual_bmis.min(), actual_bmis.max()], color='black', lw=2, label='Perfect Prediction (y=x)')
plt.xlabel('Gerçek BMI Değerleri')
plt.ylabel('Tahmin Edilen BMI Değerleri')
plt.title(f'BMI Tahmin Performansı (R2 = {r2:.4f}, RMSE = {rmse:.4f})')
plt.legend()
plt.grid(True)
plt.show()
